# Aerobic fermentor simulator · BT2026

**AI-integrated activity · Transport Phenomena Analysis**
School of Engineering and Sciences · Tecnológico de Monterrey

---

This notebook contains a web app (built with Gradio) that lets you:

1. **Determine k<sub>L</sub>·a** by the dynamic method (gassing-out) from your assigned experimental dataset.
2. **Simulate a complete aerobic fermentation** with coupled biomass/substrate/dissolved oxygen dynamics.
3. **Reverse-engineer the simulator itself** — the app makes assumptions it does not declare in its interface. Your job, aided by AI, is to find them.

## How to run this on Colab

1. `Runtime → Run all` (Ctrl+F9). It will install Gradio and launch the app.
2. Wait ~30 seconds. At the bottom of the last cell, a public URL will appear (something like `https://xxxx.gradio.live`).
3. Click that URL. The app opens in a new tab.
4. In Tab 1, upload your dataset (`.csv` file with two columns: `time (s)` and `DO (mg/L)`).
5. Read Tab 3 to understand the auditing task.

## If the URL doesn't appear

Run only the `pip install` cell first, then `Runtime → Restart session`, then run all again. That solves 90% of the cases.


In [ ]:
# Colab comes with Gradio preinstalled; this line only upgrades it if needed.
!pip install -q --upgrade gradio 2>&1 | tail -1

In [ ]:
"""
Aerobic fermentor simulator
BT2026 · Transport Phenomena Analysis
"""

import numpy as np
import pandas as pd
import gradio as gr
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# --------------------------------------------------------------
# System constants
# --------------------------------------------------------------
C_STAR = 7.5  # mg/L


# ==============================================================
# Module 1 — kLa determination by the dynamic method
# ==============================================================
def dynamic_model(t, kLa, C0):
    return C_STAR - (C_STAR - C0) * np.exp(-kLa * t)


def fit_kLa(csv_file):
    if csv_file is None:
        return None, "Upload a CSV with two columns: time (s) and DO (mg/L)."

    df = pd.read_csv(csv_file.name)
    t = df.iloc[:, 0].values.astype(float)
    C = df.iloc[:, 1].values.astype(float)
    C0 = C[0]

    def _fit(t, kLa):
        return dynamic_model(t, kLa, C0)

    try:
        popt, pcov = curve_fit(_fit, t, C, p0=[0.01], bounds=(0, 10))
        kLa_s = popt[0]
        kLa_h = kLa_s * 3600
        stderr_s = np.sqrt(np.diag(pcov))[0]
        stderr_h = stderr_s * 3600

        residuals = C - _fit(t, kLa_s)
        r_std = np.std(residuals)
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((C - np.mean(C)) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                                        gridspec_kw={"height_ratios": [3, 1]})

        ax1.scatter(t, C, label="Experimental data", color="#1a63a9", s=40, zorder=3)
        t_fit = np.linspace(t.min(), t.max(), 300)
        ax1.plot(t_fit, _fit(t_fit, kLa_s), "r-",
                 label=f"Fit: kLa = {kLa_h:.1f} 1/h", linewidth=2)
        ax1.axhline(C_STAR, color="gray", linestyle="--",
                    label=f"C* = {C_STAR} mg/L", alpha=0.7)
        ax1.set_ylabel("Dissolved oxygen (mg/L)")
        ax1.set_title("Dynamic-method fit")
        ax1.legend()
        ax1.grid(alpha=0.3)

        ax2.scatter(t, residuals, color="#c0392b", s=25)
        ax2.axhline(0, color="black", linestyle="-", linewidth=0.8)
        ax2.set_ylabel("Residuals (mg/L)")
        ax2.set_xlabel("Time (s)")
        ax2.grid(alpha=0.3)

        plt.tight_layout()

        report = f"""FIT RESULT
==========
kLa fitted     : {kLa_s:.5f} 1/s  =  {kLa_h:.2f} 1/h
Standard error : {stderr_h:.2f} 1/h
C0 detected    : {C0:.3f} mg/L
C* assumed     : {C_STAR} mg/L
R²             : {r_squared:.4f}
Residuals std  : {r_std:.4f} mg/L
Points used    : {len(t)}

INTERPRET WITH JUDGMENT:
- Is R² high AND do residuals look random (no pattern)?
- Or do residuals show systematic structure (curvature, bias)?
- Is the kLa within the expected range for your physical system?
- Do the model's assumptions (see Tab 3) apply to your actual run?
"""
        return fig, report

    except Exception as e:
        return None, f"Fit error: {e}"


# ==============================================================
# Module 2 — Aerobic fermentation simulation
# ==============================================================
def fermentor_model(t, y, mu_max, Ks, KO2, Yxs, qO2_max, kLa):
    X, S, C = y
    S = max(S, 0.0)
    C = max(C, 0.0)

    mu = mu_max * (S / (Ks + S)) * (C / (KO2 + C))
    qO2 = qO2_max * (C / (KO2 + C))

    dXdt = mu * X
    dSdt = -(1.0 / Yxs) * mu * X
    dCdt = kLa * (C_STAR - C) - qO2 * X

    return [dXdt, dSdt, dCdt]


def simulate_fermentor(mu_max_h, Ks, KO2, Yxs, qO2_max_h, kLa_h,
                       X0, S0, C0, t_final_h):
    mu_max_s = mu_max_h / 3600
    qO2_max_s = qO2_max_h / 3600
    kLa_s = kLa_h / 3600
    t_final_s = t_final_h * 3600

    sol = solve_ivp(
        fermentor_model,
        [0, t_final_s],
        [X0, S0, C0],
        args=(mu_max_s, Ks, KO2, Yxs, qO2_max_s, kLa_s),
        t_eval=np.linspace(0, t_final_s, 500),
        method="LSODA",
        rtol=1e-6, atol=1e-9,
    )

    if not sol.success:
        return None, f"Integrator failed: {sol.message}"

    t_h = sol.t / 3600
    X, S, C = sol.y

    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)

    axes[0].plot(t_h, X, "-", color="#1a63a9", linewidth=2)
    axes[0].set_ylabel("Biomass X (g/L)")
    axes[0].set_title("Aerobic fermentation simulation")
    axes[0].grid(alpha=0.3)

    axes[1].plot(t_h, S, "-", color="#27ae60", linewidth=2)
    axes[1].set_ylabel("Substrate S (g/L)")
    axes[1].grid(alpha=0.3)

    axes[2].plot(t_h, C, "-", color="#c0392b", linewidth=2, label="DO (mg/L)")
    axes[2].axhline(C_STAR, color="gray", linestyle="--",
                    label=f"C* = {C_STAR}", alpha=0.7)
    axes[2].axhline(0.5, color="orange", linestyle=":",
                    label="Limiting threshold (0.5 mg/L)", alpha=0.7)
    axes[2].set_ylabel("Dissolved oxygen (mg/L)")
    axes[2].set_xlabel("Time (h)")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()

    C_min = C.min()
    t_C_min = t_h[C.argmin()]
    X_max = X.max()
    t_X_max = t_h[X.argmax()]
    limiting = "YES (O2 limiting during the run)" if C_min < 0.5 else "NO"

    report = f"""SIMULATION RESULT
=================
Simulated duration : {t_final_h:.1f} h
Xmax               : {X_max:.3f} g/L (at t = {t_X_max:.2f} h)
S final            : {S[-1]:.3f} g/L
C minimum          : {C_min:.3f} mg/L (at t = {t_C_min:.2f} h)
Was O2 limiting?   : {limiting}

Parameters used:
  mu_max  = {mu_max_h} 1/h
  Ks      = {Ks} g/L
  KO2     = {KO2} mg/L
  Yxs     = {Yxs} g biomass / g substrate
  qO2_max = {qO2_max_h} mg O2 / (g biomass · h)
  kLa     = {kLa_h} 1/h
  C*      = {C_STAR} mg/L

REFLECT:
- Do the profiles make physical sense?
- At what point does the system become oxygen-transfer limited?
- If you increased kLa 3x, how would the biomass curve change? Why?
- Does the model assume something that is NOT true for your real fermentation?
"""
    return fig, report


# ==============================================================
# Gradio interface
# ==============================================================
CSS_CUSTOM = """
.gradio-container { max-width: 1100px !important; margin: auto; }
"""

with gr.Blocks(title="BT2026 · Fermentor simulator", css=CSS_CUSTOM) as demo:
    gr.Markdown("""
    # Aerobic fermentor simulator · BT2026
    Didactic tool: determine k_L·a by the dynamic method and simulate the
    full fermentation with dissolved oxygen.

    **Don't just use it — audit it.** Go to the "Reverse engineering" tab for instructions.
    """)

    with gr.Tab("① kLa determination"):
        gr.Markdown("Upload the CSV of your dataset (columns: `time (s)`, `DO (mg/L)`).")
        with gr.Row():
            with gr.Column(scale=1):
                csv_input = gr.File(label="CSV file", file_types=[".csv"])
                btn_fit = gr.Button("Fit k_L·a", variant="primary", size="lg")
            with gr.Column(scale=1):
                report_kLa = gr.Textbox(label="Report", lines=14)
        plot_kLa = gr.Plot(label="Fit and residuals")
        btn_fit.click(fit_kLa, inputs=csv_input, outputs=[plot_kLa, report_kLa])

    with gr.Tab("② Fermentation simulation"):
        gr.Markdown("""Adjust the parameters and run the full simulation. The k_L·a you
        determined in Tab 1 is entered manually below.""")
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Kinetic parameters**")
                mu_max_in = gr.Slider(0.05, 1.0, value=0.35, step=0.01, label="μ_max (1/h)")
                Ks_in = gr.Slider(0.01, 5.0, value=0.5, step=0.01, label="Ks (g/L)")
                KO2_in = gr.Slider(0.001, 1.0, value=0.05, step=0.001,
                                   label="K_O2 (mg/L)")
                Yxs_in = gr.Slider(0.1, 0.8, value=0.4, step=0.01,
                                   label="Y_xs (g biomass/g substrate)")
                qO2_max_in = gr.Slider(50, 500, value=200, step=10,
                                       label="q_O2,max (mg O2 / g biomass·h)")
            with gr.Column():
                gr.Markdown("**Transfer and initial conditions**")
                kLa_in = gr.Slider(1, 500, value=60, step=1, label="k_L·a (1/h)")
                X0_in = gr.Slider(0.01, 5.0, value=0.1, step=0.01, label="X0 (g/L)")
                S0_in = gr.Slider(1, 50, value=20, step=1, label="S0 (g/L)")
                C0_in = gr.Slider(0, 7.5, value=7.5, step=0.1, label="C0 (mg/L)")
                t_final_in = gr.Slider(1, 48, value=12, step=1, label="Final time (h)")
        btn_sim = gr.Button("Simulate fermentation", variant="primary", size="lg")
        plot_sim = gr.Plot(label="X, S, C profiles")
        report_sim = gr.Textbox(label="Simulation report", lines=18)
        btn_sim.click(
            simulate_fermentor,
            inputs=[mu_max_in, Ks_in, KO2_in, Yxs_in, qO2_max_in, kLa_in,
                    X0_in, S0_in, C0_in, t_final_in],
            outputs=[plot_sim, report_sim],
        )

    with gr.Tab("③ Reverse engineering"):
        gr.Markdown("""
        ## Your auditing task

        This simulator **makes assumptions it does not declare in the interface**.
        Your job, together with your team and supported by AI, is to find them.

        ### Guiding questions

        **On Tab 1 (kLa):**
        1. What is the fundamental equation the fit starts from?
        2. What does that equation assume about the DO sensor?
        3. What does it assume about oxygen consumption during the run?
        4. What does it assume about C*? Where did the value 7.5 mg/L come from?

        **On Tab 2 (full simulation):**
        5. What kinetic model does it use for growth? What does that model assume?
        6. Is there a maintenance term in the substrate balance? Should there be?
        7. Does k_L·a stay constant throughout the whole simulation? Is this realistic?
        8. What happens if your fermentor does NOT operate in water at 30°C, 1 atm?

        ### Auditing challenge by dataset

        Each team has a dataset with a particular characteristic. Identify which
        **specific assumption of the simulator** makes its model NOT apply directly
        to your dataset. Document the violated assumption, how you detected it,
        and what modification to the model would correct it.

        ### AI use in this activity

        You are practicing two of the five AI Moments of the course:

        - **M2 (AI as technical peer):** it helps you read code you may not master
          in detail. Ask it to explain the `fermentor_model` function line by line,
          or to compare pure Monod vs. Monod with inhibition.
        - **M3 (AI as system to audit):** ask it about the assumptions this code
          makes. Contrast the answer with what YOU see reading the code directly.
          Consult at least two different AIs and compare their answers.

        ### Deliverable

        An annex to the team's report with:
        - All the simulator assumptions you identified (with evidence in the code).
        - The assumption violated by your assigned dataset and your proposed correction.
        - Record of the interaction with the two AIs consulted (prompts and answers),
          noting any discrepancies found.
        """)


# ==============================================================
# Launch the app
# ==============================================================
demo.launch(share=True, debug=False)


---

## Final note

When you're done working with the app, **stop the kernel** (menu Runtime → Interrupt execution) to release the Gradio public URL. The `.gradio.live` URL is temporary and expires automatically after 72 hours.

If you want to share the app with the class, share the `.gradio.live` URL that appeared at the bottom of the code cell — but make sure the notebook stays open in your Colab or the URL will die.

---

*Simulator designed as part of the BT2026 seed course. Experimental data are synthetic but disciplinarily plausible. The simulator's hardcoded assumptions are deliberate and are part of the students' auditing exercise.*
